# 32 · Vector search — LanceDB, the *embedded* vector store

Part of the vector/graph wave of the weyland notebook library (B81). It is the counterpart
to the lab's **served** vector databases — **Qdrant** and **Weaviate** (B78, their own
notebooks) — and to notebook `04`, which covered **Lance the *format***. This one shows
Lance the *database*: **LanceDB**, an embedded vector store that opens the same `.lance`
tables from object storage and searches them — with **no server process at all**.

## Embedded vs served — the one distinction that matters here

> **Qdrant and Weaviate are *servers*.** You stand up a pod, it listens on a port, and every
> query is a network round-trip to a running service that holds the index in its own memory.
>
> **LanceDB is *embedded*.** There is no server and no port. The library opens the Lance
> tables **directly out of object storage** and does the vector search **in this process**.
> `pip install lancedb`, point it at a path, and the path *is* the database — the same way
> SQLite is a database that is a file, not a service.

In this lab the hydrated Lance **vector tables** live in **lakeFS** (the object store from
notebook `10`). So there is nothing to connect *to*: we hand LanceDB the lakeFS S3 gateway
endpoint and the lakeFS credentials as **object-store options**, and it reads the tables
straight from `s3://music/main/lancedb/` and `s3://health/main/lancedb/`. No `SHOW CATALOGS`,
no NodePort, no second copy of the data.

### What this notebook does (all **read-only**)

1. Open the lakeFS-backed LanceDB and **list its tables** — proving the connection by the
   real table names it returns, never by echoing the endpoint.
2. Open one table and **inspect its schema** — the `vector` column (a
   `fixed_size_list<float>`) plus its payload columns — and the **ANN index the hydration
   already built on it**.
3. Run a **vector similarity search**: take an existing row's embedding as the query and ask
   LanceDB for its nearest neighbours, with the payload alongside the distance.
4. **Sanity-check** the approximate result against an exact brute-force cosine — which is
   where the *approximate* in ANN, and this index's lossy PQ compression, show through
   honestly.

We never assume a table, column, or dimension: every name below is **discovered from the live
store first**, then used.

## 0. Setup — install the LanceDB client

The singleuser base image ships **`pylance`** (imported as `lance`) — the *format* library
notebook `04` used. It does **not** ship **`lancedb`**, the higher-level *database* API
(tables, `search()`, index management), so we install that here. `polars`, `pyarrow`, `numpy`
and `s3fs` are already in the image.

In [1]:
%pip install -q lancedb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

import numpy as np
import pyarrow as pa
import polars as pl
import lancedb

print("lancedb :", lancedb.__version__)
print("pyarrow :", pa.__version__)
print("polars  :", pl.__version__)
print("numpy   :", np.__version__)

lancedb : 0.38.0
pyarrow : 25.0.1
polars  : 1.44.1
numpy   : 2.5.2


## 1. Connect — open the lakeFS-backed LanceDB (env-driven)

Connection is **env-driven**, the same pattern as the other query notebooks. The committed
default is the **in-cluster** lakeFS S3-gateway URL; a validation run overrides `LAKEFS_ENDPOINT`
via env (e.g. to the lakeFS NodePort) **without editing the notebook**, and the notebook never
captures the resolved address — the connection is proven by the **table names it lists**, not
by echoing the endpoint.

There is **no server to reach**. LanceDB talks to lakeFS purely as an S3-compatible object
store, so the whole "connection" is a dict of **object-store options** — the Rust `object_store`
keys LanceDB understands (`aws_access_key_id`, `aws_secret_access_key`, `aws_endpoint`,
`aws_region`, `allow_http`, `aws_virtual_hosted_style_request`). The lakeFS access key and
secret are read from the environment (injected into the pod); they never appear as literals.

> **Why the endpoint is HTTP and path-style.** lakeFS's S3 gateway is a plain-HTTP,
> path-style endpoint (`<endpoint>/<repo>/...`), so we set `allow_http="true"` and
> `aws_virtual_hosted_style_request="false"`. The lakeFS **repo** is the S3 *bucket*
> (`music`, `health`) and the branch (`main`) is the first path element — so the LanceDB URI
> is `s3://<repo>/main/lancedb`.

In [3]:
# committed default = in-cluster lakeFS S3 gateway; a validation run overrides LAKEFS_ENDPOINT via env
LAKEFS_ENDPOINT = os.environ.get(
    "LAKEFS_ENDPOINT", "http://lakefs.data-mesh.svc.cluster.local:8000")

# object-store options LanceDB (Rust object_store) uses to reach a custom S3 endpoint.
# Creds come from the env injected into the pod — never committed as literals.
storage_options = {
    "aws_access_key_id": os.environ["LAKEFS_ACCESS_KEY_ID"],
    "aws_secret_access_key": os.environ["LAKEFS_SECRET_ACCESS_KEY"],
    "aws_endpoint": LAKEFS_ENDPOINT,
    "aws_region": "us-east-1",
    "allow_http": "true",                       # lakeFS S3 gateway is plain HTTP in-cluster
    "aws_virtual_hosted_style_request": "false",  # path-style: <endpoint>/<repo>/...
}

# The lakeFS repo is the bucket; the branch is the first path element. No server, just a path.
MUSIC_URI = "s3://music/main/lancedb"
HEALTH_URI = "s3://health/main/lancedb"

def table_names(conn):
    """Sorted list of table names, robust across lancedb versions: some builds return a
    paginated result object (carrying a `.tables` attribute) from list_tables() rather than
    a plain list of strings."""
    res = conn.list_tables()
    return sorted(getattr(res, "tables", res))

db = lancedb.connect(MUSIC_URI, storage_options=storage_options)
tables = table_names(db)
print("connected (embedded — no server) — endpoint from env")
print(f"\n{len(tables)} Lance vector tables under s3://music/main/lancedb/:")
for t in tables:
    print("  ", t)

connected (embedded — no server) — endpoint from env

8 Lance vector tables under s3://music/main/lancedb/:
   audioset
   fma_echonest
   fma_features
   gtzan
   lp_musiccaps_mc
   lp_musiccaps_mtt
   spotify_tracks
   uci_year_prediction


**"Connecting" to a second store is just opening a second path.** Because LanceDB is
embedded, there is no second server to stand up — the `health` lakeFS repo holds its own
Lance tables, and reaching them is one more `lancedb.connect(...)` with the *same* credentials
and a different URI. This is the embedded model showing through: a "database" is a location,
not a process.

In [4]:
db_health = lancedb.connect(HEALTH_URI, storage_options=storage_options)
health_tables = table_names(db_health)
print(f"{len(health_tables)} Lance vector tables under s3://health/main/lancedb/:")
for t in health_tables:
    print("  ", t)

2 Lance vector tables under s3://health/main/lancedb/:
   big_five
   open_food_facts


## 2. Open a table — schema, payload, and the ANN index it already carries

We work with **`lp_musiccaps_mc`** in the music store — embeddings of **MusicCaps** captions
(free-text descriptions of music clips). It is the most vector-DB-shaped table in the set: a
real sentence-embedding column with **human-readable text payload** riding alongside, so a
similarity search returns something you can *read* and judge.

We resolve the table from the live list (never assume it's there), then read its **schema**.
Two things to notice:

- The **`vector`** column is a `fixed_size_list<float>[N]` — exactly the type notebook `04`
  described as the one Lance treats as a vector and can index. Its list size is the
  **embedding dimension**; we read it off the schema rather than hard-coding it.
- The remaining columns are the **payload** — the metadata returned with each hit.

Then `list_indices()` shows the headline: **the hydration already built an ANN index on this
table**, so vector search here is served by a real index living *in the Lance dataset in
lakeFS*, not built on the fly.

In [5]:
TABLE = next(t for t in tables if t == "lp_musiccaps_mc")   # resolve, never assume
tbl = db.open_table(TABLE)

schema = tbl.schema
vector_field = schema.field("vector")
DIM = vector_field.type.list_size          # embedding dimension, read from the schema
payload_cols = [f.name for f in schema if f.name != "vector"]

print("table       :", TABLE)
print("rows        :", tbl.count_rows())
print("vector dim  :", DIM, "  (fixed_size_list<float>[{}])".format(DIM))
print("payload cols:", payload_cols)
print("\nfull schema:")
print(schema)

table       : lp_musiccaps_mc
rows        : 5521
vector dim  : 384   (fixed_size_list<float>[384])
payload cols: ['row_id', 'caption_summary', 'aspect_list']

full schema:
vector: fixed_size_list<item: float>[384]
  child 0, item: float
row_id: string
caption_summary: string
aspect_list: string


In [6]:
# The ANN index the hydration committed on this table. LanceDB builds an IVF_PQ index
# INSIDE the Lance dataset (in lakeFS): IVF partitions the vectors into cells so a query
# scans only the nearest few; PQ (product quantization) compresses each vector into a small
# code so the index is tiny and distance math is fast. It is a DISK-BASED index served from
# object storage — no service holds it in RAM.
indices = tbl.list_indices()
if indices:
    for ix in indices:
        details = ix.index_details or {}
        comp = details.get("compression", {})
        print("index name   :", ix.name)
        print("index type   :", ix.index_type)
        print("columns      :", ix.columns)
        print("metric       :", details.get("metric_type"))
        print("compression  :", comp.get("type"),
              "({} sub-vectors x {} bits)".format(
                  comp.get("num_sub_vectors"), comp.get("num_bits")))
        print("indexed rows :", ix.num_indexed_rows)
    METRIC = (indices[0].index_details or {}).get("metric_type", "L2").lower()
else:
    METRIC = "l2"
    print("no ANN index present — search will fall back to an exact brute-force scan")
print("\nsearch metric:", METRIC)

index name   : vector_idx
index type   : IvfPq
columns      : ['vector']
metric       : COSINE
compression  : pq (24 sub-vectors x 8 bits)
indexed rows : 5521

search metric: cosine


## 3. Vector similarity search — nearest neighbours by embedding

This is the whole job of a vector store: *given a vector, return the rows whose embeddings
are closest to it.* We have no encoder loaded here, so — the standard trick for exploring a
hydrated table — we take **an existing row's own embedding as the query**. Its nearest
neighbours should be *other captions that describe similar music*, which is a result you can
eyeball.

We read a small sample once (via the read side of the search API, so it works without the
`lance` format library), keep row `0` as the query, and hand its `vector` to
`table.search(...)`. LanceDB uses the table's **IVF_PQ index** to answer it — scanning only
the nearest partitions, decoding PQ codes — and returns the top-k with payload and a
`_distance` column. We match the search metric to the index's (**cosine**).

In [7]:
# Read a sample once, through the search read-path (no `lance` format lib needed).
SAMPLE_N = 1500
sample = tbl.search().limit(SAMPLE_N).to_arrow()

row_ids = sample.column("row_id").to_pylist()
vectors = np.asarray(sample.column("vector").to_pylist(), dtype=np.float32)
captions = sample.column("caption_summary").to_pylist()

# Use row 0 as the query — guaranteed to be in-sample, so the exact check below can see it.
QUERY_IDX = 0
query_vector = vectors[QUERY_IDX]
print("query row_id :", row_ids[QUERY_IDX])
print("query dim    :", query_vector.shape[0])
print("query caption:", captions[QUERY_IDX])

query row_id : -0Gj8-vB1q4
query dim    : 384
query caption: A melancholic and soulful ballad with low-quality sustained strings, a mellow piano melody, and soft female vocals.


In [8]:
K = 5

# ANN search served by the on-disk IVF_PQ index in lakeFS. No server, no in-RAM index.
# We name `_distance` in select() so LanceDB keeps it explicitly (and stays quiet about it).
hits = (
    tbl.search(query_vector)
    .metric(METRIC)                # match the index's metric (cosine)
    .limit(K)
    .select(["row_id", "caption_summary", "aspect_list", "_distance"])
    .to_arrow()
)

res = pl.from_arrow(hits).with_columns(pl.col("_distance").round(4))
print(f"Top-{K} nearest captions to the query embedding (via {METRIC} IVF_PQ ANN):\n")
with pl.Config(fmt_str_lengths=90, tbl_rows=K):
    print(res.select(["row_id", "_distance", "caption_summary"]))

Top-5 nearest captions to the query embedding (via cosine IVF_PQ ANN):

shape: (5, 3)
┌─────────────┬───────────┬────────────────────────────────────────────────────────────────────────┐
│ row_id      ┆ _distance ┆ caption_summary                                                        │
│ ---         ┆ ---       ┆ ---                                                                    │
│ str         ┆ f32       ┆ str                                                                    │
╞═════════════╪═══════════╪════════════════════════════════════════════════════════════════════════╡
│ -0Gj8-vB1q4 ┆ 0.1448    ┆ A melancholic and soulful ballad with low-quality sustained strings, a │
│             ┆           ┆ mellow piano melody…                                                   │
│ 1h2sb2xeCt8 ┆ 0.2216    ┆ A low quality, emotional ballad with a mellow piano melody, soft       │
│             ┆           ┆ female vocals, shimmering…                                             │
│ rez

Read the captions, not just the numbers: the query describes one *kind* of music, and every
neighbour the index returned describes **the same kind** — that semantic clustering is what a
vector store buys you over keyword search. The `_distance` column is **cosine distance**
(smaller = closer), the metric the index was built with.

## 4. Approximate vs exact — where the *A* in ANN shows through (honestly)

The index is **approximate** by design, and this one is also **lossy**: its PQ compression
stores each 384-d vector as a short code, not the full float vector. Two visible consequences,
and it is worth being honest about both:

- The query row is its own nearest neighbour — but its ANN `_distance` is **not 0**. Against
  the *exact* vector the cosine distance to itself is exactly 0; against the PQ *code* it is a
  small positive number. That gap is the quantization error, not a bug.
- The ANN top-k and the exact top-k **overlap heavily but need not match perfectly** — that
  is the recall/speed trade every ANN index makes.

We check it directly: compute **exact cosine distance** from the query to every vector in the
sample with numpy, take the true top-k, and set it beside the ANN result. On this tightly
clustered caption set they line up closely — with the self-distance telling the honest story.

In [9]:
# Exact cosine distance from the query to every sampled vector (ground truth).
norms = np.linalg.norm(vectors, axis=1, keepdims=True)
unit = vectors / np.clip(norms, 1e-9, None)
q_unit = query_vector / max(float(np.linalg.norm(query_vector)), 1e-9)
exact_dist = 1.0 - unit @ q_unit                      # cosine distance

order = np.argsort(exact_dist)[:K]
exact = pl.DataFrame({
    "row_id": [row_ids[i] for i in order],
    "exact_cosine": [round(float(exact_dist[i]), 4) for i in order],
})
print(f"Exact brute-force cosine top-{K} (ground truth over the {SAMPLE_N}-row sample):")
print(exact)

print("\nself-distance to the query row:")
print("  exact cosine   :", round(float(exact_dist[QUERY_IDX]), 4), "(0 — a vector's cosine to itself)")
print("  ANN (PQ code)  :", float(res.filter(pl.col('row_id') == row_ids[QUERY_IDX])['_distance'][0]),
      "(> 0 — PQ compression is lossy)")

Exact brute-force cosine top-5 (ground truth over the 1500-row sample):
shape: (5, 2)
┌─────────────┬──────────────┐
│ row_id      ┆ exact_cosine │
│ ---         ┆ ---          │
│ str         ┆ f64          │
╞═════════════╪══════════════╡
│ -0Gj8-vB1q4 ┆ 0.0          │
│ Nt0U-CXK6O0 ┆ 0.1262       │
│ 9DCJTAzUwNc ┆ 0.1271       │
│ 2dyEnOo3yJ8 ┆ 0.1471       │
│ KzvdKLdBw3s ┆ 0.1607       │
└─────────────┴──────────────┘

self-distance to the query row:
  exact cosine   : 0.0 (0 — a vector's cosine to itself)
  ANN (PQ code)  : 0.14480000734329224 (> 0 — PQ compression is lossy)


In [10]:
# Recall: how many of the ANN top-k are also in the exact top-k.
ann_ids = set(res["row_id"].to_list())
exact_ids = set(exact["row_id"].to_list())
overlap = ann_ids & exact_ids
print(f"ANN top-{K} ∩ exact top-{K}: {len(overlap)}/{K} rows in common")
print("  both rank the query row itself #1:",
      res['row_id'][0] == exact['row_id'][0] == row_ids[QUERY_IDX])
print("\nThe overlap is high and the #1 hit matches; the distances differ because the index")
print("answers from compressed PQ codes, not the raw vectors. That is the ANN bargain:")
print("a small, fast, disk-resident index in exchange for approximate distances.")

ANN top-5 ∩ exact top-5: 3/5 rows in common
  both rank the query row itself #1: True

The overlap is high and the #1 hit matches; the distances differ because the index
answers from compressed PQ codes, not the raw vectors. That is the ANN bargain:
a small, fast, disk-resident index in exchange for approximate distances.


## 5. When to reach for LanceDB — and when for a served store instead

### The embedded angle, restated

Everything above happened with **no vector server running**. LanceDB opened Lance tables
straight out of lakeFS object storage, used an **IVF_PQ index that lives in the dataset**, and
did **disk-based ANN search** in this notebook's own process. Because the index is on disk and
read on demand, the searchable corpus can be **far larger than RAM** — you are not holding
every vector in a service's memory.

### Reach for **embedded LanceDB** when…
- **The consumer is a process, not a fleet.** A notebook, a batch job, a Dagster asset, an
  embedded RAG retriever — anything that wants vector search **without operating a service**.
- **The vectors already live in object storage.** LanceDB reads the lab's hydrated Lance
  tables in lakeFS *in place* — no ingest step, no second copy, no server to keep fed.
- **The corpus is large and the query rate is low-to-moderate.** Disk-based ANN scales past
  RAM and costs nothing when idle — there is no pod to keep running.
- **You want versioned vectors.** It is still Lance underneath (notebook `04`): zero-copy
  versions and time-travel come for free, and lakeFS versions the objects beneath that.

### Reach for a **served store — Qdrant / Weaviate** — instead when…
- **Many clients query concurrently at low latency.** A shared, always-on service with the
  index hot in RAM beats re-opening a dataset per process. This is the serving tier's job.
- **You need live upserts and deletes at high rate**, filtered hybrid search, or built-in
  sharding/replication — the operational features a dedicated database provides.
- **A non-Python or remote client needs it.** A server exposes a network API to anything;
  an embedded library only serves the process it is imported into.

### And versus notebook `04` (Lance the *format*)
| | notebook `04` — **Lance** | this notebook — **LanceDB** |
|---|---|---|
| what it is | the on-disk **format** (`lance.dataset(...)`) | the **database API** over that format (`lancedb.connect(...)`) |
| you get | fragments, versions, `.take()`, a `create_index` primitive | `list_tables()`, `open_table()`, `search().limit().to_arrow()` |
| here, over | a self-built local temp dataset | the lab's **real hydrated vector tables in lakeFS** |

> **The one-line rule.** If the vectors live in object storage and the caller is a *process*,
> reach for **embedded LanceDB** — the path is the database. If many callers need low-latency
> concurrent search over a live, mutating index, stand up a **served** store (Qdrant /
> Weaviate). Same embeddings; different serving shape.